# Paso 11: validación por paciente (punto A4)

Rehace la tabla principal del artículo separando **registros completos** de DS1 para la validación, en lugar de repartir latidos de los mismos registros entre entrenamiento y validación.

Qué cambia respecto a lo anterior:

- Cuatro registros de DS1 se apartan enteros. Los otros dieciocho entrenan **todos** los modelos, incluidos los de árboles, para que la comparación siga siendo equitativa.
- La detención temprana y la búsqueda de los pesos de decisión usan esos cuatro registros, de modo que el ajuste ya no se apoya en pacientes vistos durante el entrenamiento.
- Se incluye la configuración con ajuste de umbral, para comprobar si ahora sí transfiere a DS2.
- Se calculan intervalos de confianza por remuestreo de registros.

**Antes de ejecutar:** activa la **T4 GPU**. Tarda entre una hora y hora y cuarto. No cierres la pestaña.

## 1. Instalar y descargar datos

In [ ]:
!pip install wfdb xgboost -q
import wfdb, time, json
import numpy as np, pandas as pd
from scipy.signal import lfilter
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, recall_score, precision_score, confusion_matrix
from xgboost import XGBClassifier

wfdb.dl_database('mitdb', dl_dir='mitdb')
print('Descarga lista')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 3.9 MB/s eta 0:00:00
Generating record list for: 100
Generating record list for: 101
Generating record list for: 102
Generating record list for: 103
Generating record list for: 104
Generating record list for: 105
Generating record list for: 106
Generating record list for: 107
Generating record list for: 108
Generating record list for: 109
Generating record list for: 111
Generating record list for: 112
Generating record list for: 113
Generating record list for: 114
Generating record list for: 115
Generating record list for: 116
Generating record list for: 117
Generating record list for: 118
Generating record list for: 119
Generating record list for: 121
Generating record list for: 122
Generating record list for: 123
Generating record list for: 124
Generating record list for: 200
Generating record list for: 201
Generating record list for: 202
Generating record list for: 203
Generating record list for: 205
Generating record list f

## 2. Preparar los latidos, guardando el registro de cada uno

In [ ]:
DS1 = [101,106,108,109,112,114,115,116,118,119,122,124,201,203,205,207,208,209,215,220,223,230]
DS2 = [100,103,105,111,113,117,121,123,200,202,210,212,213,214,219,221,222,228,231,232,233,234]
AAMI = {'N':'N','L':'N','R':'N','e':'N','j':'N','A':'S','a':'S','J':'S','S':'S','V':'V','E':'V','F':'F'}
CODIGOS_LATIDO = set('NLRejAaJSVEF/fQ')
CLASES = ['N','S','V','F']
FS, ANTES, DESPUES = 360, 90, 162

def filtro_pasa_altas(x, fs=FS, fc=1.0):
    alpha = np.exp(-2*np.pi*fc/fs)
    return x - lfilter([1-alpha], [1, -alpha], x)

def extraer_latidos(num):
    ruta = f'mitdb/{num}'
    rec = wfdb.rdrecord(ruta); ann = wfdb.rdann(ruta, 'atr')
    canal = rec.sig_name.index('MLII') if 'MLII' in rec.sig_name else 0
    x = filtro_pasa_altas(rec.p_signal[:, canal]); x = (x - x.mean())/x.std()
    pos = np.array([p for p, s in zip(ann.sample, ann.symbol) if s in CODIGOS_LATIDO])
    sim = [s for s in ann.symbol if s in CODIGOS_LATIDO]
    rr = np.diff(pos)/FS; rr_medio = rr.mean()
    V, R, Y, G = [], [], [], []
    for i in range(1, len(pos)-1):
        if sim[i] not in AAMI: continue
        r = pos[i]
        if r-ANTES < 0 or r+DESPUES > len(x): continue
        V.append(x[r-ANTES : r+DESPUES])
        R.append([rr[i-1], rr[i], rr[max(0,i-10):i].mean(), rr[i-1]/rr_medio, rr[i]/rr_medio])
        Y.append(AAMI[sim[i]]); G.append(int(num))
    return V, R, Y, G

def construir(lista):
    V, R, Y, G = [], [], [], []
    for n in lista:
        v, r, y, g = extraer_latidos(n); V += v; R += r; Y += y; G += g
    return (np.array(V, dtype='float32'), np.array(R, dtype='float32'),
            np.array(Y), np.array(G))

V_ent, R_ent, y_ent, reg_ent = construir(DS1)
V_pru, R_pru, y_pru, reg_pru = construir(DS2)
a_numero = {c:i for i,c in enumerate(CLASES)}
y_ent_num = np.array([a_numero[c] for c in y_ent])
y_pru_num = np.array([a_numero[c] for c in y_pru])
print('DS1:', len(y_ent_num), 'latidos |', len(set(reg_ent)), 'registros')
print('DS2:', len(y_pru_num), 'latidos |', len(set(reg_pru)), 'registros')

DS1: 50969 latidos | 22 registros
DS2: 49661 latidos | 22 registros


## 3. Separar cuatro registros completos para validación

In [ ]:
# Validacion por paciente: se apartan 4 registros COMPLETOS de DS1.
# Se eligen de forma deterministica, exigiendo que las cuatro clases esten
# representadas, para que el ajuste de umbral pueda estimarse sobre todas ellas.
N_VAL = 4
MIN_POR_CLASE = 15
rng_sel = np.random.default_rng(42)
registros_ds1 = sorted(set(reg_ent))

eleccion, mejor = None, -1
for _ in range(500):
    cand = sorted(rng_sel.choice(registros_ds1, N_VAL, replace=False).tolist())
    m = np.isin(reg_ent, cand)
    cuenta = [int((y_ent_num[m] == i).sum()) for i in range(4)]
    if min(cuenta) >= MIN_POR_CLASE:
        eleccion = cand; break
    if min(cuenta) > mejor:
        mejor, eleccion = min(cuenta), cand

val_reg = eleccion
es_val = np.isin(reg_ent, val_reg)
V_tr, R_tr, y_tr = V_ent[~es_val], R_ent[~es_val], y_ent_num[~es_val]
V_val, R_val, y_val = V_ent[es_val], R_ent[es_val], y_ent_num[es_val]

print('Registros de validacion:', val_reg)
print('Registros de entrenamiento:', [r for r in registros_ds1 if r not in val_reg])
print()
print('Latidos  entrenamiento:', {c: int((y_tr == a_numero[c]).sum()) for c in CLASES}, '| total', len(y_tr))
print('Latidos  validacion   :', {c: int((y_val == a_numero[c]).sum()) for c in CLASES}, '| total', len(y_val))

Registros de validacion: [112, 124, 205, 223]
Registros de entrenamiento: [np.int64(101), np.int64(106), np.int64(108), np.int64(109), np.int64(114), np.int64(115), np.int64(116), np.int64(118), np.int64(119), np.int64(122), np.int64(201), np.int64(203), np.int64(207), np.int64(208), np.int64(209), np.int64(215), np.int64(220), np.int64(230)]

Latidos  entrenamiento: {'N': 37143, 'S': 834, 'V': 3197, 'F': 384} | total 41558
Latidos  validacion   : {'N': 8681, 'S': 109, 'V': 591, 'F': 30} | total 9411


## 4. Preparativos comunes

In [ ]:
# Todos los modelos se entrenan con los MISMOS 18 registros, para que la
# comparacion siga siendo equitativa. Los 4 apartados solo sirven para la
# detencion temprana y para elegir los pesos de decision.
X_tr  = np.hstack([R_tr, V_tr[:, ::10]])
X_val = np.hstack([R_val, V_val[:, ::10]])
X_pru = np.hstack([R_pru, V_pru[:, ::10]])

SEMILLAS = [0, 1, 2, 3, 4]
filas, predicciones = [], {}

def registrar(etiqueta, semilla, pred, extra=None):
    se = recall_score(y_pru_num, pred, labels=range(4), average=None, zero_division=0)
    pp = precision_score(y_pru_num, pred, labels=range(4), average=None, zero_division=0)
    f1 = f1_score(y_pru_num, pred, labels=range(4), average=None, zero_division=0)
    fila = {'modelo': etiqueta, 'semilla': semilla,
            'F1_macro_4': f1.mean(), 'F1_macro_3': f1[:3].mean()}
    for i, c in enumerate(CLASES):
        fila[f'Se_{c}'] = se[i]; fila[f'+P_{c}'] = pp[i]
    if extra: fila.update(extra)
    filas.append(fila)
    predicciones[(etiqueta, semilla)] = pred
    print(f'  {etiqueta} (semilla {semilla}): F1(4)={fila["F1_macro_4"]:.3f}  '
          f'F1(3)={fila["F1_macro_3"]:.3f}  Se_S={se[1]:.3f}  +P_S={pp[1]:.3f}')

## 5. Modelos de árboles

In [ ]:
pesos_v = compute_class_weight('balanced', classes=np.arange(4), y=y_tr)
peso_muestra = pesos_v[y_tr]

print('Random Forest')
for s in SEMILLAS:
    m = RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=-1, random_state=s)
    m.fit(X_tr, y_tr)
    inicio = time.time(); pred = m.predict(X_pru)
    registrar('Random Forest', s, pred, {'ms': (time.time()-inicio)*1000/len(X_pru),
                                         'tamano': sum(t.tree_.node_count for t in m.estimators_)})

print('XGBoost')
for s in SEMILLAS:
    m = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, tree_method='hist',
                      objective='multi:softprob', num_class=4, random_state=s, n_jobs=-1)
    m.fit(X_tr, y_tr, sample_weight=peso_muestra)
    inicio = time.time(); pred = m.predict(X_pru)
    registrar('XGBoost', s, pred, {'ms': (time.time()-inicio)*1000/len(X_pru),
                                   'tamano': len(m.get_booster().trees_to_dataframe())})

Random Forest
  Random Forest (semilla 0): F1(4)=0.452  F1(3)=0.601  Se_S=0.020  +P_S=0.837
  Random Forest (semilla 1): F1(4)=0.447  F1(3)=0.594  Se_S=0.020  +P_S=0.860
  Random Forest (semilla 2): F1(4)=0.449  F1(3)=0.599  Se_S=0.020  +P_S=0.900
  Random Forest (semilla 3): F1(4)=0.448  F1(3)=0.598  Se_S=0.021  +P_S=0.864
  Random Forest (semilla 4): F1(4)=0.449  F1(3)=0.597  Se_S=0.020  +P_S=0.902
XGBoost
  XGBoost (semilla 0): F1(4)=0.503  F1(3)=0.655  Se_S=0.124  +P_S=0.326
  XGBoost (semilla 1): F1(4)=0.503  F1(3)=0.655  Se_S=0.124  +P_S=0.326
  XGBoost (semilla 2): F1(4)=0.503  F1(3)=0.655  Se_S=0.124  +P_S=0.326
  XGBoost (semilla 3): F1(4)=0.503  F1(3)=0.655  Se_S=0.124  +P_S=0.326
  XGBoost (semilla 4): F1(4)=0.503  F1(3)=0.655  Se_S=0.124  +P_S=0.326


## 6. Red convolucional, con y sin ajuste de umbral

Los pesos de decisión se buscan sobre los cuatro registros apartados, que ahora corresponden a pacientes distintos de los del entrenamiento.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

pesos_clase = {i: w for i, w in enumerate(pesos_v)}

def armar_cnn():
    ef = keras.Input(shape=(252, 1))
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(ef)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 3, activation='relu', padding='same')(x)
    return ef, layers.GlobalAveragePooling1D()(x)

def armar_bilstm():
    ef = keras.Input(shape=(252, 1))
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(ef)
    return ef, layers.Bidirectional(layers.LSTM(32))(x)

def armar(constructor):
    ef, x = constructor()
    er = keras.Input(shape=(5,))
    z = layers.concatenate([x, er])
    z = layers.Dense(64, activation='relu')(z)
    z = layers.Dropout(0.3)(z)
    s = layers.Dense(4, activation='softmax')(z)
    m = keras.Model([ef, er], s)
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005, clipnorm=1.0),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

REJILLA_S = np.round(np.arange(0.2, 3.01, 0.2), 2)
REJILLA_V = np.round(np.arange(0.4, 2.01, 0.2), 2)

def buscar_pesos(prob_val):
    mejor, mejor_f1 = (1.0, 1.0), -1.0
    for ws in REJILLA_S:
        for wv in REJILLA_V:
            p = (prob_val * np.array([1.0, ws, wv, 1.0])).argmax(1)
            f1 = f1_score(y_val, p, labels=range(4), average='macro', zero_division=0)
            if f1 > mejor_f1: mejor_f1, mejor = f1, (ws, wv)
    return mejor

def correr_red(nombre, constructor, con_umbral=None):
    for s in SEMILLAS:
        keras.utils.set_random_seed(s)
        m = armar(constructor)
        m.fit([V_tr[..., None], R_tr], y_tr,
              validation_data=([V_val[..., None], R_val], y_val),
              epochs=40, batch_size=256, class_weight=pesos_clase,
              callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
                         keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4)],
              verbose=0)
        inicio = time.time()
        prob_pru = m.predict([V_pru[..., None], R_pru], verbose=0)
        ms = (time.time()-inicio)*1000/len(V_pru)
        registrar(nombre, s, prob_pru.argmax(1), {'ms': ms, 'tamano': m.count_params()})
        if con_umbral:
            prob_val = m.predict([V_val[..., None], R_val], verbose=0)
            ws, wv = buscar_pesos(prob_val)
            pred = (prob_pru * np.array([1.0, ws, wv, 1.0])).argmax(1)
            registrar(con_umbral, s, pred, {'ms': ms, 'tamano': m.count_params(),
                                            'peso_S': ws, 'peso_V': wv})

print('CNN 1D')
correr_red('CNN 1D', armar_cnn, con_umbral='CNN 1D + umbral')

CNN 1D
  CNN 1D (semilla 0): F1(4)=0.572  F1(3)=0.761  Se_S=0.716  +P_S=0.316
  CNN 1D + umbral (semilla 0): F1(4)=0.542  F1(3)=0.722  Se_S=0.876  +P_S=0.246
  CNN 1D (semilla 1): F1(4)=0.541  F1(3)=0.720  Se_S=0.537  +P_S=0.271
  CNN 1D + umbral (semilla 1): F1(4)=0.519  F1(3)=0.691  Se_S=0.609  +P_S=0.215
  CNN 1D (semilla 2): F1(4)=0.565  F1(3)=0.753  Se_S=0.422  +P_S=0.357
  CNN 1D + umbral (semilla 2): F1(4)=0.576  F1(3)=0.767  Se_S=0.713  +P_S=0.320
  CNN 1D (semilla 3): F1(4)=0.539  F1(3)=0.708  Se_S=0.521  +P_S=0.204
  CNN 1D + umbral (semilla 3): F1(4)=0.529  F1(3)=0.698  Se_S=0.643  +P_S=0.202
  CNN 1D (semilla 4): F1(4)=0.541  F1(3)=0.719  Se_S=0.534  +P_S=0.237
  CNN 1D + umbral (semilla 4): F1(4)=0.527  F1(3)=0.701  Se_S=0.716  +P_S=0.213


## 7. Red recurrente bidireccional

Esta es la parte más lenta, unos 40 minutos.

In [ ]:
print('BiLSTM')
correr_red('BiLSTM', armar_bilstm)

BiLSTM
  BiLSTM (semilla 0): F1(4)=0.370  F1(3)=0.488  Se_S=0.129  +P_S=0.053
  BiLSTM (semilla 1): F1(4)=0.303  F1(3)=0.402  Se_S=0.097  +P_S=0.027
  BiLSTM (semilla 2): F1(4)=0.368  F1(3)=0.479  Se_S=0.285  +P_S=0.072
  BiLSTM (semilla 3): F1(4)=0.358  F1(3)=0.473  Se_S=0.101  +P_S=0.061
  BiLSTM (semilla 4): F1(4)=0.355  F1(3)=0.464  Se_S=0.195  +P_S=0.044


## 8. Tabla principal

In [ ]:
df = pd.DataFrame(filas)
orden = ['Random Forest', 'XGBoost', 'CNN 1D', 'CNN 1D + umbral', 'BiLSTM']
orden = [o for o in orden if o in set(df['modelo'])]
cols = ['F1_macro_4','F1_macro_3'] + [f'Se_{c}' for c in CLASES] + [f'+P_{c}' for c in CLASES]
res = df.groupby('modelo')[cols].agg(['mean','std']).reindex(orden)

tabla = pd.DataFrame(index=res.index)
for c in cols:
    tabla[c] = [f'{a:.3f} ± {b:.3f}' for a, b in zip(res[(c,'mean')], res[(c,'std')])]
tabla['tamano'] = df.groupby('modelo')['tamano'].first().reindex(orden).values
tabla['ms'] = df.groupby('modelo')['ms'].mean().reindex(orden).round(3).values

pd.set_option('display.width', 260)
print('TABLA PRINCIPAL CON VALIDACION POR PACIENTE (DS2, 5 ejecuciones)')
print(tabla.T)
df.to_csv('a4_por_ejecucion.csv', index=False)
tabla.to_csv('a4_resumen.csv')

if 'peso_S' in df.columns:
    print()
    print('Pesos de decision elegidos sobre los registros de validacion:')
    print(df[df['peso_S'].notna()][['semilla','peso_S','peso_V']].to_string(index=False))

TABLA PRINCIPAL CON VALIDACION POR PACIENTE (DS2, 5 ejecuciones)
modelo      Random Forest        XGBoost         CNN 1D CNN 1D + umbral         BiLSTM
F1_macro_4  0.449 ± 0.002  0.503 ± 0.000  0.551 ± 0.016   0.538 ± 0.022  0.351 ± 0.027
F1_macro_3  0.597 ± 0.003  0.655 ± 0.000  0.732 ± 0.023   0.716 ± 0.031  0.461 ± 0.034
Se_N        0.982 ± 0.001  0.933 ± 0.000  0.919 ± 0.026   0.882 ± 0.027  0.719 ± 0.043
Se_S        0.020 ± 0.000  0.124 ± 0.000  0.546 ± 0.106   0.711 ± 0.103  0.161 ± 0.079
Se_V        0.829 ± 0.002  0.893 ± 0.000  0.943 ± 0.005   0.948 ± 0.007  0.768 ± 0.041
Se_F        0.002 ± 0.001  0.157 ± 0.000  0.014 ± 0.025   0.010 ± 0.016  0.097 ± 0.026
+P_N        0.942 ± 0.000  0.956 ± 0.000  0.974 ± 0.006   0.982 ± 0.004  0.962 ± 0.012
+P_S        0.873 ± 0.028  0.326 ± 0.000  0.277 ± 0.061   0.239 ± 0.048  0.051 ± 0.017
+P_V        0.758 ± 0.014  0.795 ± 0.000  0.845 ± 0.045   0.792 ± 0.057  0.356 ± 0.059
+P_F        0.080 ± 0.143  0.028 ± 0.000  0.006 ± 0.008   0.005 ±

## 9. Intervalos de confianza por remuestreo de registros

In [ ]:
rng = np.random.default_rng(0)
B = 1000
registros = sorted(set(reg_pru))
por_registro = {g: np.where(reg_pru == g)[0] for g in registros}

def metricas_rapidas(yr, yp):
    cm = np.bincount(yr*4+yp, minlength=16).reshape(4,4)
    vp = np.diag(cm).astype(float)
    se = np.divide(vp, cm.sum(1), out=np.zeros(4), where=cm.sum(1)>0)
    pp = np.divide(vp, cm.sum(0), out=np.zeros(4), where=cm.sum(0)>0)
    f1 = np.divide(2*se*pp, se+pp, out=np.zeros(4), where=(se+pp)>0)
    return f1.mean(), f1[:3].mean(), se[1], pp[1]

def intervalos(pred):
    acum = np.zeros((B, 4))
    for b in range(B):
        elegidos = rng.choice(registros, len(registros), replace=True)
        idx = np.concatenate([por_registro[g] for g in elegidos])
        acum[b] = metricas_rapidas(y_pru_num[idx], pred[idx])
    salida = {}
    for i, n in enumerate(['F1_macro_4','F1_macro_3','Se_S','+P_S']):
        lo, hi = np.percentile(acum[:, i], [2.5, 97.5])
        salida[n] = f'{acum[:, i].mean():.3f} [{lo:.3f}; {hi:.3f}]'
    return salida

print(f'Intervalos de confianza del 95 % por remuestreo de los {len(registros)} registros de DS2')
tabla_ic = pd.DataFrame({m: intervalos(predicciones[(m, 0)]) for m in orden}).T
print(tabla_ic.to_string())
tabla_ic.to_csv('a4_intervalos.csv')

Intervalos de confianza del 95 % por remuestreo de los 22 registros de DS2
                           F1_macro_4            F1_macro_3                  Se_S                  +P_S
Random Forest    0.461 [0.385; 0.537]  0.613 [0.512; 0.716]  0.045 [0.001; 0.203]  0.783 [0.300; 1.000]
XGBoost          0.514 [0.430; 0.609]  0.665 [0.563; 0.779]  0.219 [0.033; 0.553]  0.349 [0.122; 0.713]
CNN 1D           0.557 [0.475; 0.642]  0.742 [0.632; 0.855]  0.692 [0.398; 0.831]  0.306 [0.038; 0.724]
CNN 1D + umbral  0.528 [0.437; 0.627]  0.703 [0.583; 0.835]  0.830 [0.537; 0.909]  0.235 [0.026; 0.588]
BiLSTM           0.368 [0.301; 0.429]  0.484 [0.397; 0.564]  0.193 [0.062; 0.415]  0.056 [0.012; 0.115]


## 10. Descargar resultados

In [ ]:
from google.colab import files
for a in ['a4_resumen.csv', 'a4_por_ejecucion.csv', 'a4_intervalos.csv']:
    files.download(a)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>